In [3]:
import numpy as np
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
import os
import random
from geopy.distance import geodesic
from plotly import express as px
import json 
import plotly.graph_objects as go


## Plotting helper functions

In [24]:
# Function to extract coordinates from GeoJSON and draw boundaries
def add_geojson_boundaries(fig, geojson):
    for feature in geojson['features']:
        geometry = feature['geometry']
        coords = []
        
        if geometry['type'] == 'Polygon':
            coords = geometry['coordinates'][0]  # Outer boundary
        elif geometry['type'] == 'MultiPolygon':
            for polygon in geometry['coordinates']:
                coords.extend(polygon[0])
                coords.append([None, None])  # Separator for discontinuous lines
        
        if coords:
            lons, lats = zip(*coords)
            fig.add_trace(go.Scattergeo(
                lon=lons,
                lat=lats,
                mode='lines',
                line=dict(width=1, color='lightblue'),
                showlegend=False,
                hoverinfo='skip'
            ))


# National population (by state)

In [ ]:
mexico_states = {
    "Aguascalientes": "01","Baja California": "02", "Baja California Sur": "03","Campeche": "04", "Coahuila de Zaragoza": "05","Colima": "06", "Chiapas": "07","Chihuahua": "08","Ciudad de México": "09","Durango": "10","Guanajuato": "11","Guerrero": "12","Hidalgo": "13","Jalisco": "14","México": "15", "Michoacán de Ocampo": "16", "Morelos": "17", "Nayarit": "18", "Nuevo León": "19", "Oaxaca": "20","Puebla": "21", "Querétaro": "22", "Quintana Roo": "23", "San Luis Potosí": "24", "Sinaloa": "25", "Sonora": "26", "Tabasco": "27", "Tamaulipas": "28", "Tlaxcala": "29", "Veracruz de Ignacio de la Llave": "30", "Yucatán": "31", "Zacatecas": "32"
}
state = 'San Luis Potosí'

filename = f'../Datos Nacionales/POBLACION/ITER_{mexico_states[state]}XLSX20.xlsx'

state_pop_df = pd.read_excel(filename)
# print(state_pop_df.head())

# Air data

In [ ]:
air = pd.read_excel('../Datos SLP/AmbientalLADRILLERAS.xlsx')
air['type'] = 'air'
air = air.rename(columns={'x': 'X', 'y':'Y'})

with open('../Datos SLP/shapefiles/24-SLP.geojson', 'r') as f:
    slp_geojson = json.load(f)





# Create figure
fig = go.Figure()

# Add GeoJSON boundaries
add_geojson_boundaries(fig, slp_geojson)

color_col = 'nombre'
# Add scatter points
for eff in air[color_col].unique():
    mun_data = air[air[color_col] == eff]
    # print(mun_data)
    fig.add_trace(go.Scattergeo(
        lon=mun_data['X'],
        lat=mun_data['Y'],
        text=[mun_data['nom_mun']],
        name=eff,
        mode='markers',
        marker=dict(size=8), 
        opacity=0.5
    ))

fig.update_geos(
    fitbounds="locations",
    visible=False
)

fig.update_layout(
    height=600,
    showlegend=True
)


# fig.update_layout(title_text='Map of ladrilleras')
# fig.show()
# fig.write_html('ladrilleras.html')

In [ ]:
retc = pd.read_excel('../Datos SLP/AmbientalRETC.xlsx')
retc['type'] = 'retc'
# print(retc.head())
# Create figure
fig = go.Figure()

# Add GeoJSON boundaries
add_geojson_boundaries(fig, slp_geojson)

color_col = 'SUST1'
# Add scatter points
for eff in retc[color_col].unique():
    mun_data = retc[retc[color_col] == eff]
    # print(mun_data)
    fig.add_trace(go.Scattergeo(
        lon=mun_data['X'],
        lat=mun_data['Y'],
        text=[mun_data['SUST1'], mun_data['empresa'], mun_data['efecto']],
        name=eff,
        mode='markers',
        marker=dict(size=8), 
        opacity=0.5
    ))

fig.update_geos(
    fitbounds="locations",
    visible=False
)

fig.update_layout(
    height=600,
    showlegend=True
)


# fig.update_layout(title_text='Map of toxins')
# fig.show()
# fig.write_html('retc.html')


## Ladrilleras

In [15]:
ladrilleras = pd.read_excel('../Datos SLP/AmbientalLADRILLERAS.xlsx')
ladrilleras.head()
ladrilleras = ladrilleras.rename(columns={'x': 'X', 'y':'Y'})
ladrilleras['type'] = 'ladrilleras'


## Minas y metales

In [21]:
minas = pd.read_excel('../Datos SLP/AmbientalMinasMetales.xlsx')
minas.head()
minas = minas.rename(columns={'Longitud_D':'X', 'Latitud_D': 'Y'})
minas['type'] = 'minas y metales'

# Overlaid map

In [ ]:



merged = pd.concat([air,retc, ladrilleras, minas], ignore_index=True)
# print(merged.head())

fig = go.Figure()
# Add GeoJSON boundaries
add_geojson_boundaries(fig, slp_geojson)

color_col = 'type'
# Add scatter points
for eff in merged[color_col].unique():
    
    mun_data = merged[merged[color_col] == eff]
    # print(mun_data)
    fig.add_trace(go.Scattergeo(
        lon=mun_data['X'],
        lat=mun_data['Y'],
        text=[mun_data['SUST1']],
        name=eff,
        mode='markers',
        marker=dict(size=8), 
        opacity=0.5
    ))


fig.update_geos(
    fitbounds="locations"
)
fig.update_layout(
    height=600,
    showlegend=True, title_text='Overlaid pollutants'
)
# fig.show()

In [ ]:
# TODO: still a work in progress, but trying to get a population heatmap
fig = go.Figure()
fig = px.choropleth(
    state_pop_df, 
    geojson=slp_geojson, 
    color='POBTOT', 
    locations='NOM_MUN',  # Replace with the column in state_pop_df that matches GeoJSON features
    featureidkey='properties.d_codigo'  # Replace with the key in GeoJSON properties
)
fig.update_geos(
    fitbounds="locations"
)
fig.update_layout(
    height=600,
    showlegend=True, title_text='SLP population heatmap'
)
# fig.show()
